# Проект. Создание рекомендательной системы. Часть 1

## Описание задачи

На этот раз вы работаете в **Яндекс Музыке** — популярном стриминговом сервисе с большим каталогом, насчитывающим более **70 миллионов треков**. Чтобы пользователям было легче ориентироваться в таком объёме музыкального контента, необходимо создать эффективную систему **персональных рекомендаций**.

Цель проекта — построить **прототип работающей рекомендательной системы**, а не добиваться максимального качества метрик. Поэтому в рамках проекта **не требуется fine-tuning моделей**, подбор гиперпараметров и других коэффициентов.

## Проблема

Взаимодействие с музыкальным сервисом должно быть удобным, простым и приятным. Для этого пользователю нужно предлагать треки или подборки, основываясь на его вкусах и предпочтениях.

Практически все популярные музыкальные сервисы используют рекомендательные системы, поскольку пользователи не хотят тратить время на самостоятельный поиск интересной музыки, если сервис может предложить релевантный контент автоматически.

## Бизнес-задача

Необходимо улучшить пользовательский опыт, создав систему персональных рекомендаций.

## Задача машинного обучения

Для решения задачи нужно использовать методы машинного обучения:

- взять за основу взаимодействия примерно **1,4 миллиона пользователей** с **1 миллионом треков**;
- построить пайплайн для расчёта персональных рекомендаций с использованием изученных ранее алгоритмов;
- разработать сервис рекомендаций;
- подготовить набор скриптов в репозитории, который позволит:
  - рассчитать персональные рекомендации;
  - запустить сервис рекомендаций;
  - получить рекомендации для пользователей.

## Данные

Данные находятся в трёх файлах.

### 1. Данные о треках — `tracks.parquet`

Содержит информацию о музыкальных треках:

- `track_id` — идентификатор музыкального трека;
- `albums` — список идентификаторов альбомов, содержащих трек;
- `artists` — список идентификаторов исполнителей трека;
- `genres` — список идентификаторов жанров, к которым принадлежит трек.

### 2. Названия каталожных сущностей — `catalog_names.parquet`

Содержит названия треков, альбомов, исполнителей и жанров:

- `id` — идентификатор одной из каталожных единиц;
- `type` — тип идентификатора;
- `name` — имя или название каталожной единицы.

### 3. История прослушиваний — `interactions.parquet`

Содержит информацию о взаимодействии пользователей с треками:

- `user_id` — идентификатор пользователя;
- `track_id` — идентификатор музыкального трека;
- `track_seq` — порядковый номер трека в истории пользователя;
- `started_at` — дата начала прослушивания трека.

## Инфраструктура и инструменты

Для выполнения проекта понадобятся следующие инструменты:

- виртуальное окружение Python;
- **Jupyter Lab**;
- **Visual Studio Code**;
- персональный **S3-бакет**.

# Инициализация

Загружаем библиотеки необходимые для выполнения кода ноутбука.

In [72]:
# pip install s3fs pyarrow implicit scikit-learn
import os
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

import logging

In [73]:
# настройки
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

%matplotlib inline
%config InlineBackend.figure_format = 'png'
%config InlineBackend.figure_format = 'retina'

# === ЭТАП 1 ===

# Загрузка первичных данных

Загружаем первичные данные из файлов:
- tracks.parquet
- catalog_names.parquet
- interactions.parquet

In [ ]:
# корневые директории
DATA_DIR = "./data"
LOCAL_OUT_DIR = "./artifacts"

os.makedirs(LOCAL_OUT_DIR, exist_ok=True)
# данные
TRACKS_PATH = os.path.join(DATA_DIR, "tracks.parquet")
CATALOG_PATH = os.path.join(DATA_DIR, "catalog_names.parquet")
INTERACTIONS_PATH = os.path.join(DATA_DIR, "interactions.parquet")

# S3 бакет
S3_BASE = "s3://your-bucket/recsys"
S3_DATA_DIR = f"{S3_BASE}/data"
S3_REC_DIR = f"{S3_BASE}/recommendations"

In [37]:
tracks = pd.read_parquet(TRACKS_PATH)
catalog = pd.read_parquet(CATALOG_PATH)
interactions = pd.read_parquet(INTERACTIONS_PATH)

print(tracks.shape, catalog.shape, interactions.shape)
display(tracks.head())
display(catalog.head())
display(interactions.head())

(1000000, 4) (1812471, 3) (222629898, 4)


,track_id,albums,artists,genres
0,26,"[3, 2490753]",[16],"[11, 21]"
1,38,"[3, 2490753]",[16],"[11, 21]"
2,135,"[12, 214, 2490809]",[84],[11]
3,136,"[12, 214, 2490809]",[84],[11]
4,138,"[12, 214, 322, 72275, 72292, 91199, 213505, 2490809, 6007655, 17294156]",[84],[11]


,id,type,name
0,3,album,Taller Children
1,12,album,Wild Young Hearts
2,13,album,Lonesome Crow
3,17,album,Graffiti Soul
4,26,album,Blues Six Pack


,user_id,track_id,track_seq,started_at
0,0,99262,1,2022-07-17
1,0,589498,2,2022-07-19
2,0,590262,3,2022-07-21
3,0,590303,4,2022-07-22
4,0,590692,5,2022-07-22


# Обзор данных

Проверяем данные, есть ли с ними явные проблемы.

In [38]:
# описание датасета tracks
tracks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 4 columns):
 #   Column    Non-Null Count    Dtype 
---  ------    --------------    ----- 
 0   track_id  1000000 non-null  int64 
 1   albums    1000000 non-null  object
 2   artists   1000000 non-null  object
 3   genres    1000000 non-null  object
dtypes: int64(1), object(3)
memory usage: 30.5+ MB


In [39]:
# основные статистики tracks
tracks.describe()

,track_id
count,1.000000e+06
mean,3.685121e+07
std,2.679771e+07
min,2.600000e+01
25%,1.543088e+07
50%,3.455047e+07
75%,5.692557e+07
max,1.015218e+08


В track_id максимум 101 521 800 млн, что намного меньше лимита int32, поэтому можно выполнить приведение типов int64 -> int32

In [40]:
# описание датасета catalog
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1812471 entries, 0 to 1812470
Data columns (total 3 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   id      int64 
 1   type    object
 2   name    object
dtypes: int64(1), object(2)
memory usage: 41.5+ MB


In [41]:
# основные статистики catalog
catalog.describe()

,id
count,1.812471e+06
mean,2.321647e+07
std,2.526044e+07
min,0.000000e+00
25%,3.480524e+06
50%,1.211436e+07
75%,3.773817e+07
max,1.015218e+08


В id максимум 101 521 800 млн, что намного меньше лимита int32, поэтому можно выполнить приведение типов int64 -> int32

In [42]:
# описание датасета interactions
interactions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 222629898 entries, 0 to 291
Data columns (total 4 columns):
 #   Column      Dtype         
---  ------      -----         
 0   user_id     int32         
 1   track_id    int32         
 2   track_seq   int16         
 3   started_at  datetime64[ns]
dtypes: datetime64[ns](1), int16(1), int32(2)
memory usage: 5.4 GB


In [43]:
# основные статистики interactions
interactions.describe()

,user_id,track_id,track_seq,started_at
count,2.226299e+08,2.226299e+08,2.226299e+08,222629898
mean,6.875767e+05,3.653622e+07,4.621403e+02,2022-08-29 16:39:44.541336320
min,0.000000e+00,2.600000e+01,1.000000e+00,2022-01-01 00:00:00
25%,3.433710e+05,1.480849e+07,5.600000e+01,2022-07-02 00:00:00
50%,6.879730e+05,3.552474e+07,1.810000e+02,2022-09-15 00:00:00
75%,1.031127e+06,5.651137e+07,5.060000e+02,2022-11-09 00:00:00
max,1.374582e+06,1.015218e+08,1.663700e+04,2022-12-31 00:00:00
std,3.969033e+05,2.661782e+07,8.257312e+02,NaN


## Оптимизация памяти

In [ ]:
# Выполним приведение типов данных, чтобы уменьшить объем требуемой памяти
# Это вероятно уменьшит память почти в 2–3 раза
tracks["track_id"] = tracks["track_id"].astype("int32")

catalog["id"] = catalog["id"].astype("int32")
catalog["type"] = catalog["type"].astype("category")

In [49]:
# Посмотрим, есть ли уменьшение памяти
display(tracks.info())
display(catalog.info())
display(interactions.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 4 columns):
 #   Column    Non-Null Count    Dtype 
---  ------    --------------    ----- 
 0   track_id  1000000 non-null  int32 
 1   albums    1000000 non-null  object
 2   artists   1000000 non-null  object
 3   genres    1000000 non-null  object
dtypes: int32(1), object(3)
memory usage: 26.7+ MB


None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1812471 entries, 0 to 1812470
Data columns (total 3 columns):
 #   Column  Dtype   
---  ------  -----   
 0   id      int32   
 1   type    category
 2   name    object  
dtypes: category(1), int32(1), object(1)
memory usage: 22.5+ MB


None

<class 'pandas.core.frame.DataFrame'>
Index: 222629898 entries, 0 to 291
Data columns (total 4 columns):
 #   Column      Dtype         
---  ------      -----         
 0   user_id     int32         
 1   track_id    int32         
 2   track_seq   int16         
 3   started_at  datetime64[ns]
dtypes: datetime64[ns](1), int16(1), int32(2)
memory usage: 5.4 GB


None

### Наблюдения за памятью после преобразований:
memory usage:

    - tracks: 30.5+ MB -> 26.7+ MB

    - catalog: 41.5+ MB -> 22.5+ MB

## Проверка пропусков во всех таблицах, оценка разреженности данных


In [65]:
def analyze_dataframe(df):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().sum() / len(df)) * 100,
        "non_null_count": df.notna().sum(),
        "memory_MB": df.memory_usage(deep=True, index=False) / (1024 ** 2)
    })

    unique_values = {}

    for col in df.columns:
        series = df[col]

        if series.dtype == "object":
            non_null = series.dropna()

            if not non_null.empty and isinstance(non_null.iloc[0], (list, np.ndarray)):
                unique_values[col] = series.explode().dropna().nunique()
            else:
                unique_values[col] = series.nunique()
        else:
            unique_values[col] = series.nunique()

    summary["n_unique"] = pd.Series(unique_values)

    summary["missing_count"] = summary["missing_count"].astype("int64")
    summary["non_null_count"] = summary["non_null_count"].astype("int64")
    summary["n_unique"] = summary["n_unique"].astype("int64")
    summary["missing_percent"] = summary["missing_percent"].round(4)
    summary["memory_MB"] = summary["memory_MB"].round(4)

    return summary.sort_values("missing_count", ascending=False)

In [67]:
# анализ датасета tracks
analyze_dataframe(tracks)

,dtype,missing_count,missing_percent,non_null_count,memory_MB,n_unique
track_id,int32,0,0.0,1000000,3.8147,1000000
albums,object,0,0.0,1000000,114.4409,658724
artists,object,0,0.0,1000000,114.4409,153581
genres,object,0,0.0,1000000,114.4409,173


### Выводы по анализу датасета tracks

В результате анализа структуры таблицы tracks были получены следующие наблюдения.

- Таблица содержит 1 000 000 записей, каждая из которых соответствует одному музыкальному треку.

- Пропущенные значения отсутствуют во всех столбцах (track_id, albums, artists, genres), поэтому дополнительная обработка пропусков не требуется.

- Столбец track_id содержит 1 000 000 уникальных значений, что подтверждает уникальность идентификатора трека.

- Столбцы albums, artists и genres имеют тип object, так как содержат списки идентификаторов, соответствующих альбомам, исполнителям и жанрам.

- Количество уникальных значений:

        - albums — 658 724 уникальных идентификатора альбомов;

        - artists — 153 581 уникальный исполнитель;

        - genres — 173 уникальных жанра.

- Основной объём памяти занимают столбцы albums, artists и genres (около 114 MB каждый), что связано с хранением списков объектов.

Таким образом, таблица tracks представляет собой каталог музыкальных треков с информацией о связанных альбомах, исполнителях и жанрах и может использоваться в дальнейшем для построения рекомендательной системы.

In [69]:
# анализ датасета catalog
analyze_dataframe(catalog)

,dtype,missing_count,missing_percent,non_null_count,memory_MB,n_unique
id,int32,0,0.0,1812471,6.9140,1776697
type,category,0,0.0,1812471,1.7289,4
name,object,0,0.0,1812471,142.5674,945118


### Выводы по анализу датасета catalog

Таблица catalog содержит справочную информацию о каталожных сущностях музыкального сервиса.

- Таблица содержит 1 812 471 записей.

- Пропущенные значения отсутствуют во всех столбцах (id, type, name), что говорит о полной заполненности справочника.

- Столбец id содержит 1 776 697 уникальных идентификаторов, которые используются в других таблицах для связи с объектами каталога.

- Столбец type имеет тип category и содержит 4 уникальных значения, соответствующих различным типам каталожных объектов (например, треки, альбомы, исполнители и жанры).

- Столбец name содержит 945 118 уникальных названий, что соответствует названиям музыкальных объектов (треков, альбомов, исполнителей и жанров).

- Основной объём памяти занимает столбец name (≈142 MB), так как он хранит строковые значения.

Таким образом, таблица catalog представляет собой справочник названий музыкальных сущностей, который используется для интерпретации идентификаторов, присутствующих в таблицах tracks и interactions.

In [74]:
# анализ датасета interactions
analyze_dataframe(interactions)

,dtype,missing_count,missing_percent,non_null_count,memory_MB,n_unique
user_id,int32,0,0.0,222629898,849.2657,1373221
track_id,int32,0,0.0,222629898,849.2657,1000000
track_seq,int16,0,0.0,222629898,424.6328,16637
started_at,datetime64[ns],0,0.0,222629898,1698.5313,365


### Выводы по анализу датасета interactions

Таблица interactions содержит информацию о взаимодействиях пользователей с музыкальными треками (события прослушивания).

- Таблица включает 222 629 898 записей, каждая из которых соответствует одному факту прослушивания трека пользователем.

- Пропущенные значения отсутствуют во всех столбцах (user_id, track_id, track_seq, started_at), поэтому дополнительная обработка пропусков не требуется.

- Количество уникальных пользователей (user_id) составляет 1 373 221, что отражает масштаб аудитории сервиса.

- В таблице представлены 1 000 000 уникальных треков, что соответствует размеру каталога треков в таблице tracks.

- Максимальное значение track_seq равно 16637, что означает, что история прослушивания отдельного пользователя может содержать более 16 тысяч треков.

- Столбец started_at содержит 365 уникальных дат, что показывает, что данные охватывают один календарный год активности пользователей.

- Наибольший объём памяти занимает столбец started_at (≈ 1.7 GB), поскольку хранит значения типа datetime.

Таким образом, таблица interactions представляет собой крупный журнал событий прослушивания пользователей и будет использоваться как основной источник данных для построения рекомендательной системы.

## Проверка неизвестных артистов / альбомов / жанров

In [ ]:
# Сначала получаем все id из каталога.
artist_ids = set(catalog[catalog.type == "artist"].id)
album_ids = set(catalog[catalog.type == "album"].id)
genre_ids = set(catalog[catalog.type == "genre"].id)

### Ответим на вопросы:
- есть ли неизвестные id?
- сколько их?
- сколько треков они затрагивают?

In [ ]:
# Проверка неизвестных id
unknown_artists = tracks["artists"].explode().dropna()
unknown_artists = unknown_artists[~unknown_artists.isin(artist_ids)]

unknown_albums = tracks["albums"].explode().dropna()
unknown_albums = unknown_albums[~unknown_albums.isin(album_ids)]

unknown_genres = tracks["genres"].explode().dropna()
unknown_genres = unknown_genres[~unknown_genres.isin(genre_ids)]

print("Unknown artists occurrences:", len(unknown_artists))
print("Unknown albums occurrences:", len(unknown_albums))
print("Unknown genres occurrences:", len(unknown_genres))

print("Unknown artists unique:", unknown_artists.nunique())
print("Unknown albums unique:", unknown_albums.nunique())
print("Unknown genres unique:", unknown_genres.nunique())

Unknown artists occurrences: 0
Unknown albums occurrences: 0
Unknown genres occurrences: 48369
Unknown artists unique: 0
Unknown albums unique: 0
Unknown genres unique: 30


In [ ]:
# Сколько треков затронуто
tracks_with_unknown_artists = tracks["artists"].apply(
    lambda x: any(pd.notna(i) and i not in artist_ids for i in x)
).sum()

tracks_with_unknown_albums = tracks["albums"].apply(
    lambda x: any(pd.notna(i) and i not in album_ids for i in x)
).sum()

tracks_with_unknown_genres = tracks["genres"].apply(
    lambda x: any(pd.notna(i) and i not in genre_ids for i in x)
).sum()

print("Tracks with unknown artists:", tracks_with_unknown_artists)
print("Tracks with unknown albums:", tracks_with_unknown_albums)
print("Tracks with unknown genres:", tracks_with_unknown_genres)

Tracks with unknown artists: 0
Tracks with unknown albums: 0
Tracks with unknown genres: 48345


# Выводы

Приведём выводы по первому знакомству с данными:
- есть ли с данными явные проблемы,
- какие корректирующие действия (в целом) были предприняты.

После оптимизации типов данных удалось уменьшить потребление памяти.

- Для таблицы tracks память сократилась с 30.5 MB до 26.7 MB 

        - благодаря приведению track_id к типу int32

- Для таблицы catalog память уменьшилась с 41.5 MB до 22.5 MB 

        - благодаря приведению id к типу int32

        - за счёт преобразования столбца type из object в category

# === ЭТАП 2 ===

# EDA

Распределение количества прослушанных треков.

Наиболее популярные треки

Наиболее популярные жанры

Треки, которые никто не прослушал

# Преобразование данных

Преобразуем данные в формат, более пригодный для дальнейшего использования в расчётах рекомендаций.

# Сохранение данных

Сохраним данные в двух файлах в персональном S3-бакете по пути `recsys/data/`:
- `items.parquet` — все данные о музыкальных треках,
- `events.parquet` — все данные о взаимодействиях.

# Очистка памяти

Здесь, может понадобится очистка памяти для высвобождения ресурсов для выполнения кода ниже. 

Приведите соответствующие код, комментарии, например:
- код для удаление более ненужных переменных,
- комментарий, что следует перезапустить kernel, выполнить такие-то начальные секции и продолжить с этапа 3.

# === ЭТАП 3 ===

# Загрузка данных

Если необходимо, то загружаем items.parquet, events.parquet.

# Разбиение данных

Разбиваем данные на тренировочную, тестовую выборки.

# Топ популярных

Рассчитаем рекомендации как топ популярных.

# Персональные

Рассчитаем персональные рекомендации.

# Похожие

Рассчитаем похожие, они позже пригодятся для онлайн-рекомендаций.

# Построение признаков

Построим три признака, можно больше, для ранжирующей модели.

# Ранжирование рекомендаций

Построим ранжирующую модель, чтобы сделать рекомендации более точными. Отранжируем рекомендации.

# Оценка качества

Проверим оценку качества трёх типов рекомендаций: 

- топ популярных,
- персональных, полученных при помощи ALS,
- итоговых
  
по четырем метрикам: recall, precision, coverage, novelty.

# === Выводы, метрики ===

Основные выводы при работе над расчётом рекомендаций, рассчитанные метрики.